[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/060.%20The%20Warehouse%20Slotting%20Optimization%20Problem/P60-Tier-9_executed.ipynb)

---


# 60. The Warehouse Slotting Optimization Problem

## Tier 9: Quantum Optimization Concepts and Classical Simulation

This notebook develops a binary quadratic model for warehouse slotting, simulates an annealing-style search on a classical computer, and compares its decoded assignment with a greedy baseline, a hybrid local-refinement procedure, and a population-based quantum-inspired search. The sequence is: formulate the QUBO, search for low-energy binary states, decode assignments, compare methods, and interpret the results.

The code does **not** run on quantum hardware. Its annealing routine is a classical simulated-annealing-style algorithm; its population-based routine is a genetic search with quantum-inspired labels, not an implementation of QAOA or VQE. Consequently, their measured runtimes and solution values cannot demonstrate quantum speedup or quantum advantage.

### Goal, inputs, and outputs

- **Goal:** assign SKUs to warehouse locations to maximize a velocity-and-zone score while penalizing assignment violations and pairwise capacity conflicts.
- **Inputs:** generated SKU attributes (including picking velocity, space requirement, and weight) and location attributes (including capacity, weight limit, accessibility, and zone preference).
- **Outputs:** assignments, QUBO energies, objective values, feasibility checks, runtime measurements for these implementations, and plots of the recorded comparisons.
- **Scope:** this is a teaching model, not a complete warehouse-planning system. It does not model travel paths, replenishment, inventory levels, labor, time-varying demand, or the detailed operational meaning of storage zones.

### Model assumptions and limitations

Each SKU is intended to receive exactly one location. The QUBO penalizes pairs of SKUs whose combined space requirement or weight exceeds a location's limit. These are **pairwise conflict penalties**, not exact aggregate-capacity constraints: a set of three or more individually compatible SKUs can exceed a location's capacity even when no pair does. Therefore, a low QUBO energy does not by itself certify that the decoded assignment satisfies total location capacities. The notebook reports capacity feasibility separately.

The greedy baseline enforces aggregate space and weight limits as it builds its assignment. The local-refinement routine checks aggregate limits for each proposed destination, but its starting assignment comes from decoding the QUBO and is not guaranteed to be feasible. Interpret comparisons together with assignment counts and feasibility checks, rather than treating objective value alone as a measure of operational quality.

### Method outline

1. **QUBO formulation:** represent each possible SKU-location assignment with a binary variable and combine the score with penalty terms.
2. **Energy interpretation:** treat lower QUBO energy as preferable; the annealing simulation searches for a low-energy state.
3. **Classical baseline:** build a feasible assignment greedily, prioritizing high-velocity SKUs and accessible locations.
4. **Hybrid procedure:** decode an annealing result and try feasible, score-improving single-SKU moves.
5. **Quantum-inspired population search:** evolve a population using tournament selection, crossover, and mutation. This is a classical genetic search, not QAOA.
6. **Comparison and interpretation:** compare objective values, run times, energy histories, assignment counts, and aggregate-capacity feasibility.

### What to look for

A lower QUBO energy is meaningful only relative to the specified penalty model. The decoded objective measures the velocity-and-zone score, while the feasibility check tests aggregate location limits. Runtime measurements describe these particular classical implementations and the computer on which they run; they are not evidence of quantum performance. Results can vary with the generated instance and random search trajectory, although the fixed random seeds make a run reproducible in a consistent software environment.


### QUBO model

A binary decision variable records whether SKU \(i\) is assigned to location \(j\):

$$
x_{ij} \in \{0,1\}, \qquad i \in \mathcal{I},\; j \in \mathcal{J}.
$$

Here, \(\mathcal{I}\) is the set of SKUs and \(\mathcal{J}\) is the set of locations. The value \(x_{ij}=1\) means that SKU \(i\) uses location \(j\); otherwise it is zero. Define the score for that assignment by

$$
s_{ij} = \frac{v_i}{a_j} + 10p_j,
$$

where \(v_i\) is SKU velocity, \(a_j\) is location accessibility, and \(p_j\) is the location's zone preference. The first term rewards placing faster-moving SKUs in more accessible locations. The second term rewards locations with a higher zone-preference value.

The model minimizes the following energy:

$$
E(x) = -\sum_{i \in \mathcal{I}}\sum_{j \in \mathcal{J}} s_{ij}x_{ij}
+ A\sum_{i \in \mathcal{I}}\left(\sum_{j \in \mathcal{J}}x_{ij}-1\right)^2
+ B\sum_{(i,k,j)\in\mathcal{C}}x_{ij}x_{kj}.
$$

The negative sign turns score maximization into energy minimization. The assignment penalty, weighted by \(A\), is zero when each SKU has exactly one selected location and positive when its number of selected locations differs from one. The set \(\mathcal{C}\) contains pairs of distinct SKUs whose combined space requirement exceeds a location's capacity or whose combined weight exceeds its weight limit. The conflict penalty \(B\) discourages selecting both members of such a pair at that location. In the code, \(A\) is `penalty_weight`, and the pairwise conflict penalty is a multiple of that weight.

For binary variables, \(x_{ij}^2=x_{ij}\). Expanding the assignment penalty therefore gives a linear contribution of \(-A x_{ij}\) for each assignment variable and a quadratic contribution of \(2A x_{ij}x_{ik}\) for each pair of different locations \(j\) and \(k\) for the same SKU. The constant \(A\) per SKU is omitted because adding a constant to every candidate state's energy cannot change which state has the lowest energy. The code stores each quadratic coefficient once in the upper triangle of `qubo_matrix`; `_calculate_energy` uses that same convention.

The conflict set encodes pairwise conflicts, not the full inequalities \(\sum_i \text{space}_i x_{ij} \leq \text{capacity}_j\) and \(\sum_i \text{weight}_i x_{ij} \leq \text{weight limit}_j\). Those aggregate inequalities are checked separately when assignments are evaluated. This distinction is essential when interpreting a QUBO solution.

With \(n=|\mathcal{I}||\mathcal{J}|\) assignment variables, a dense QUBO matrix requires \(O(n^2)\) storage. The simulation stores that matrix and also caches its nonzero interactions so that each proposed single-bit flip can use only the variables connected to that bit. The annealing run makes a fixed number of single-bit proposals per read; the population search evaluates its population over multiple generations. These are computational costs of the classical implementations, not claims about quantum-computer complexity.


In [ ]:
from typing import Any, Dict, List, Optional, Tuple
import random
import time

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8")

# Fixed seeds make generated instances and stochastic searches reproducible.
np.random.seed(42)
random.seed(42)

In [ ]:
class QuantumSlottingOptimizer:
    """Build and search a slotting QUBO using classical simulations."""

    def __init__(self, num_skus: int = 200, num_locations: int = 50):
        if num_skus <= 0 or num_locations <= 0:
            raise ValueError("The number of SKUs and locations must be positive.")

        print(
            f"Initializing slotting model: {num_skus} SKUs, "
            f"{num_locations} locations"
        )

        self.skus = self._create_skus(num_skus)
        self.locations = self._create_locations(num_locations)

        # These settings control a classical annealing-style simulation; they
        # are not settings sent to a quantum device.
        self.quantum_params = {
            "annealing_time": 100,
            "temperature_schedule": "geometric",
            "num_reads": 1000,
            "chain_strength": 2.0,
        }

        self.qubo_matrix: Optional[np.ndarray] = None
        self._qubo_edges: List[Tuple[int, int, float]] = []
        self._qubo_neighbors: List[List[Tuple[int, float]]] = []
        self.quantum_solution: Optional[Dict[str, Any]] = None
        self.classical_solution: Optional[Dict[str, Any]] = None
        self.hybrid_solution: Optional[Dict[str, Any]] = None

        print(
            f"Slotting model initialized with {len(self.skus)} SKUs and "
            f"{len(self.locations)} locations"
        )

    def _create_skus(self, num_skus: int) -> List[Dict[str, Any]]:
        """Generate SKU attributes for the illustrative instance."""
        categories = [
            "Electronics",
            "Clothing",
            "Food",
            "Books",
            "Toys",
            "Sports",
            "Home",
            "Beauty",
        ]
        skus = []

        for i in range(num_skus):
            category = str(np.random.choice(categories))

            if category == "Food":
                velocity = np.random.lognormal(4.5, 0.8)
            elif category == "Electronics":
                velocity = np.random.lognormal(3.5, 1.2)
            else:
                velocity = np.random.lognormal(3.0, 1.0)

            skus.append(
                {
                    "id": i + 1,
                    "velocity": max(1.0, float(velocity)),
                    "space_req": float(np.random.uniform(0.1, 4.0)),
                    "weight": float(np.random.uniform(0.1, 50.0)),
                    "category": category,
                    "price": float(np.random.uniform(5.0, 500.0)),
                    "quantum_id": i,
                    "compatibility_group": int(np.random.randint(0, 5)),
                }
            )

        return skus

    def _create_locations(self, num_locations: int) -> List[Dict[str, Any]]:
        """Generate location attributes for the illustrative instance."""
        zones = [
            "Fast-Pick",
            "Reserve",
            "Cold-Storage",
            "High-Security",
            "Bulk",
        ]
        zone_preferences = {
            "Fast-Pick": 1.0,
            "Reserve": 0.6,
            "Cold-Storage": 0.8,
            "High-Security": 0.4,
            "Bulk": 0.3,
        }
        locations = []

        for i in range(num_locations):
            zone = str(
                np.random.choice(zones, p=[0.3, 0.4, 0.1, 0.1, 0.1])
            )

            if zone == "Fast-Pick":
                capacity = float(np.random.uniform(8, 20))
                accessibility = float(np.random.uniform(1, 3))
            elif zone == "Reserve":
                capacity = float(np.random.uniform(15, 40))
                accessibility = float(np.random.uniform(4, 8))
            elif zone == "Cold-Storage":
                capacity = float(np.random.uniform(10, 25))
                accessibility = float(np.random.uniform(3, 6))
            else:
                capacity = float(np.random.uniform(20, 50))
                accessibility = float(np.random.uniform(5, 10))

            locations.append(
                {
                    "id": i + 1,
                    "capacity": capacity,
                    "weight_limit": capacity * 25.0,
                    "accessibility": accessibility,
                    "zone": zone,
                    "quantum_id": i,
                    "zone_preference": zone_preferences[zone],
                }
            )

        return locations

    def formulate_qubo(self) -> np.ndarray:
        """Build the upper-triangular QUBO coefficient matrix."""
        print("Formulating the QUBO matrix...")

        num_skus = len(self.skus)
        num_locations = len(self.locations)
        total_vars = num_skus * num_locations
        penalty_weight = 1000.0
        qubo = np.zeros((total_vars, total_vars), dtype=float)

        # Linear terms reward high velocity and preferred zones. The negative
        # sign is required because the QUBO search minimizes energy.
        for i, sku in enumerate(self.skus):
            for j, location in enumerate(self.locations):
                var_idx = i * num_locations + j
                score = (
                    sku["velocity"] / location["accessibility"]
                    + 10.0 * location["zone_preference"]
                )
                qubo[var_idx, var_idx] = -score - penalty_weight

        # Expanding A * (sum_j x_ij - 1)^2 gives -A on each
        # diagonal and +2A on each upper-triangular within-SKU pair.
        for i in range(num_skus):
            sku_vars = [i * num_locations + j for j in range(num_locations)]
            for first in range(num_locations):
                for second in range(first + 1, num_locations):
                    qubo[sku_vars[first], sku_vars[second]] += (
                        2.0 * penalty_weight
                    )

        # Add pairwise conflict penalties. Each pair is stored once in the
        # upper triangle, matching the energy calculation below.
        for j, location in enumerate(self.locations):
            for i1 in range(num_skus):
                sku1 = self.skus[i1]
                var1 = i1 * num_locations + j

                if (
                    sku1["space_req"] > location["capacity"]
                    or sku1["weight"] > location["weight_limit"]
                ):
                    qubo[var1, var1] += 10.0 * penalty_weight

                for i2 in range(i1 + 1, num_skus):
                    sku2 = self.skus[i2]
                    space_conflict = (
                        sku1["space_req"] + sku2["space_req"]
                        > location["capacity"]
                    )
                    weight_conflict = (
                        sku1["weight"] + sku2["weight"]
                        > location["weight_limit"]
                    )

                    if space_conflict:
                        qubo[var1, i2 * num_locations + j] += (
                            5.0 * penalty_weight
                        )
                    if weight_conflict:
                        qubo[var1, i2 * num_locations + j] += (
                            5.0 * penalty_weight
                        )

        # Cache interactions so energy evaluations and single-bit proposals
        # do not repeatedly scan the entire dense matrix.
        self._qubo_edges = []
        self._qubo_neighbors = [[] for _ in range(total_vars)]
        for first in range(total_vars):
            nonzero_columns = np.flatnonzero(qubo[first, first + 1 :])
            for offset in nonzero_columns:
                second = first + 1 + int(offset)
                coefficient = float(qubo[first, second])
                self._qubo_edges.append((first, second, coefficient))
                self._qubo_neighbors[first].append((second, coefficient))
                self._qubo_neighbors[second].append((first, coefficient))

        self.qubo_matrix = qubo
        print(
            f"QUBO matrix created with {total_vars} binary variables and "
            f"{len(self._qubo_edges)} nonzero pairwise interactions"
        )
        return qubo

    def quantum_annealing_simulation(self) -> Dict[str, Any]:
        """Search the QUBO with a classical annealing-style simulation."""
        print("\n=== CLASSICAL ANNEALING-STYLE SIMULATION ===")

        if self.qubo_matrix is None:
            self.formulate_qubo()

        start_time = time.time()
        num_reads = self.quantum_params["num_reads"]
        annealing_time = self.quantum_params["annealing_time"]
        best_energy = float("inf")
        best_solution = None
        energy_history = []

        for _ in range(num_reads):
            current_solution = self._simulate_quantum_state()
            current_energy = self._calculate_energy(current_solution)
            if current_energy < best_energy:
                best_energy = current_energy
                best_solution = current_solution.copy()
            energy_history.append(best_energy)

        elapsed_time = time.time() - start_time
        assignment = self._decode_solution(best_solution)
        results = {
            "solution": assignment,
            "energy": best_energy,
            "time": elapsed_time,
            "energy_history": energy_history,
            "num_reads": num_reads,
            "annealing_time": annealing_time,
            "method": "Classical annealing-style simulation",
        }
        self.quantum_solution = results

        print(f"Simulation completed in {elapsed_time:.3f} seconds")
        print(f"Best QUBO energy: {best_energy:.2f}")
        print(f"Decoded assignments: {len(assignment)} SKUs")
        return results

    def _simulate_quantum_state(self) -> np.ndarray:
        """Run single-bit Metropolis moves under a cooling schedule."""
        num_vars = self.qubo_matrix.shape[0]
        num_locations = len(self.locations)
        num_steps = 100
        temperature_start = 10.0
        temperature_end = 0.01

        # Starting with one selected location per SKU makes the initial state
        # interpretable as an assignment while still allowing QUBO moves.
        state = np.zeros(num_vars, dtype=int)
        for i in range(len(self.skus)):
            location_index = np.random.randint(0, num_locations)
            state[i * num_locations + location_index] = 1

        for step in range(num_steps):
            fraction = step / max(num_steps - 1, 1)
            temperature = temperature_start * (
                temperature_end / temperature_start
            ) ** fraction
            variable = int(np.random.randint(0, num_vars))
            bit = state[variable]
            direction = 1 - 2 * bit

            delta_energy = float(self.qubo_matrix[variable, variable]) * direction
            for neighbor, coefficient in self._qubo_neighbors[variable]:
                delta_energy += coefficient * state[neighbor] * direction

            if delta_energy <= 0 or np.random.random() < np.exp(
                -delta_energy / temperature
            ):
                state[variable] = 1 - bit

        return state

    def _calculate_energy(self, solution: np.ndarray) -> float:
        """Evaluate diagonal and upper-triangular QUBO terms."""
        if self.qubo_matrix is None:
            raise RuntimeError("Formulate the QUBO before evaluating energy.")
        energy = float(np.dot(solution, np.diag(self.qubo_matrix) * solution))
        for first, second, coefficient in self._qubo_edges:
            energy += coefficient * solution[first] * solution[second]
        return energy

    def _decode_solution(self, binary_solution: np.ndarray) -> Dict[int, int]:
        """Choose one selected location per SKU from a binary state."""
        assignment = {}
        num_locations = len(self.locations)

        for i, sku in enumerate(self.skus):
            start = i * num_locations
            end = start + num_locations
            selected = np.flatnonzero(binary_solution[start:end])

            if len(selected) == 0:
                # Decoding guarantees one location per SKU even when a search
                # state violates the QUBO's assignment penalty.
                selected_index = int(np.argmax(self._location_scores(sku)))
            else:
                scores = self._location_scores(sku)
                selected_index = int(selected[np.argmax(scores[selected])])

            assignment[sku["id"]] = selected_index + 1

        return assignment

    def _location_scores(self, sku: Dict[str, Any]) -> np.ndarray:
        """Return the modeled score for assigning one SKU to each location."""
        return np.array(
            [
                sku["velocity"] / location["accessibility"]
                + 10.0 * location["zone_preference"]
                for location in self.locations
            ],
            dtype=float,
        )

    def classical_baseline(self) -> Dict[str, Any]:
        """Build a capacity-feasible assignment with a greedy heuristic."""
        print("\n=== GREEDY CLASSICAL BASELINE ===")
        start_time = time.time()
        assignment = {}
        location_usage = {}
        sorted_skus = sorted(
            self.skus, key=lambda sku: sku["velocity"], reverse=True
        )
        sorted_locations = sorted(
            self.locations, key=lambda location: location["accessibility"]
        )

        for sku in sorted_skus:
            for location in sorted_locations:
                loc_id = location["id"]
                usage = location_usage.setdefault(
                    loc_id, {"space": 0.0, "weight": 0.0}
                )
                if (
                    usage["space"] + sku["space_req"] <= location["capacity"]
                    and usage["weight"] + sku["weight"]
                    <= location["weight_limit"]
                ):
                    assignment[sku["id"]] = loc_id
                    usage["space"] += sku["space_req"]
                    usage["weight"] += sku["weight"]
                    break

        elapsed_time = time.time() - start_time
        results = {
            "solution": assignment,
            "objective": self._calculate_objective(assignment),
            "time": elapsed_time,
            "method": "Greedy heuristic",
            "feasibility": self._check_assignment_feasibility(assignment),
        }
        self.classical_solution = results
        print(f"Greedy construction completed in {elapsed_time:.3f} seconds")
        print(f"Objective value: {results['objective']:.2f}")
        print(f"Assigned SKUs: {len(assignment)}")
        return results

    def _calculate_objective(self, assignment: Dict[int, int]) -> float:
        """Sum the velocity-and-zone score for the supplied assignment."""
        sku_by_id = {sku["id"]: sku for sku in self.skus}
        location_by_id = {
            location["id"]: location for location in self.locations
        }
        total_score = 0.0

        for sku_id, loc_id in assignment.items():
            sku = sku_by_id[sku_id]
            location = location_by_id[loc_id]
            total_score += (
                sku["velocity"] / location["accessibility"]
                + 10.0 * location["zone_preference"]
            )

        return total_score

    def _check_assignment_feasibility(
        self, assignment: Dict[int, int]
    ) -> Dict[str, Any]:
        """Check coverage and aggregate space and weight limits."""
        location_by_id = {
            location["id"]: location for location in self.locations
        }
        sku_by_id = {sku["id"]: sku for sku in self.skus}
        usage = {
            location["id"]: {"space": 0.0, "weight": 0.0}
            for location in self.locations
        }

        for sku_id, loc_id in assignment.items():
            if sku_id not in sku_by_id or loc_id not in location_by_id:
                raise ValueError("Assignment contains an unknown SKU or location.")
            usage[loc_id]["space"] += sku_by_id[sku_id]["space_req"]
            usage[loc_id]["weight"] += sku_by_id[sku_id]["weight"]

        space_violations = []
        weight_violations = []
        for location in self.locations:
            loc_id = location["id"]
            if usage[loc_id]["space"] > location["capacity"] + 1e-9:
                space_violations.append(loc_id)
            if usage[loc_id]["weight"] > location["weight_limit"] + 1e-9:
                weight_violations.append(loc_id)

        return {
            "all_skus_assigned": len(assignment) == len(self.skus),
            "assigned_sku_count": len(assignment),
            "space_feasible": not space_violations,
            "weight_feasible": not weight_violations,
            "space_violation_locations": space_violations,
            "weight_violation_locations": weight_violations,
            "feasible": (
                len(assignment) == len(self.skus)
                and not space_violations
                and not weight_violations
            ),
        }

    def hybrid_quantum_classical(self) -> Dict[str, Any]:
        """Refine a decoded annealing assignment with feasible local moves."""
        print("\n=== HYBRID ANNEALING-STYLE AND CLASSICAL REFINEMENT ===")
        start_time = time.time()
        annealing_results = self.quantum_annealing_simulation()
        refined_assignment = self._local_refinement(
            annealing_results["solution"]
        )
        elapsed_time = time.time() - start_time
        results = {
            "solution": refined_assignment,
            "objective": self._calculate_objective(refined_assignment),
            "time": elapsed_time,
            "quantum_energy": annealing_results["energy"],
            "method": "Annealing-style search with local refinement",
            "feasibility": self._check_assignment_feasibility(
                refined_assignment
            ),
        }
        self.hybrid_solution = results
        print(f"Hybrid procedure completed in {elapsed_time:.3f} seconds")
        print(f"Objective value: {results['objective']:.2f}")
        print(f"Assigned SKUs: {len(refined_assignment)}")
        return results

    def _local_refinement(
        self, initial_assignment: Dict[int, int]
    ) -> Dict[int, int]:
        """Accept score-improving single-SKU moves with feasible destinations."""
        assignment = initial_assignment.copy()
        improved = True
        iterations = 0
        sku_by_id = {sku["id"]: sku for sku in self.skus}
        location_by_id = {
            location["id"]: location for location in self.locations
        }

        while improved and iterations < 100:
            improved = False
            iterations += 1

            for sku_id, current_loc_id in list(assignment.items()):
                sku = sku_by_id[sku_id]
                current_location = location_by_id[current_loc_id]
                current_score = (
                    sku["velocity"] / current_location["accessibility"]
                    + 10.0 * current_location["zone_preference"]
                )

                for location in self.locations:
                    if location["id"] == current_loc_id:
                        continue

                    new_score = (
                        sku["velocity"] / location["accessibility"]
                        + 10.0 * location["zone_preference"]
                    )
                    if (
                        new_score > current_score
                        and self._is_move_feasible(
                            sku_id,
                            current_loc_id,
                            location["id"],
                            assignment,
                        )
                    ):
                        assignment[sku_id] = location["id"]
                        improved = True
                        break

        return assignment

    def _is_move_feasible(
        self,
        sku_id: int,
        from_loc_id: int,
        to_loc_id: int,
        assignment: Dict[int, int],
    ) -> bool:
        """Check aggregate target limits after moving one SKU."""
        del from_loc_id  # The source can only become less loaded after removal.
        sku_by_id = {sku["id"]: sku for sku in self.skus}
        location_by_id = {
            location["id"]: location for location in self.locations
        }
        sku = sku_by_id[sku_id]
        target = location_by_id[to_loc_id]
        current_space = 0.0
        current_weight = 0.0

        for other_sku_id, other_loc_id in assignment.items():
            if other_sku_id != sku_id and other_loc_id == to_loc_id:
                other_sku = sku_by_id[other_sku_id]
                current_space += other_sku["space_req"]
                current_weight += other_sku["weight"]

        return (
            current_space + sku["space_req"] <= target["capacity"]
            and current_weight + sku["weight"] <= target["weight_limit"]
        )

    def quantum_inspired_algorithm(self) -> Dict[str, Any]:
        """Run a classical genetic search over binary QUBO states."""
        print("\n=== QUANTUM-INSPIRED POPULATION SEARCH ===")
        if self.qubo_matrix is None:
            self.formulate_qubo()

        start_time = time.time()
        population_size = 50
        generations = 100
        population = [
            self._generate_random_solution() for _ in range(population_size)
        ]
        best_solution = None
        best_energy = float("inf")
        energy_history = []

        for _ in range(generations):
            energies = [self._calculate_energy(solution) for solution in population]
            best_index = int(np.argmin(energies))
            if energies[best_index] < best_energy:
                best_energy = energies[best_index]
                best_solution = population[best_index].copy()
            energy_history.append(best_energy)

            new_population = []
            for _ in range(population_size):
                parent1 = self._tournament_selection(population, energies)
                parent2 = self._tournament_selection(population, energies)
                child = self._quantum_crossover(parent1, parent2)
                child = self._quantum_mutation(child)
                new_population.append(child)
            population = new_population

        elapsed_time = time.time() - start_time
        assignment = self._decode_solution(best_solution)
        results = {
            "solution": assignment,
            "objective": self._calculate_objective(assignment),
            "energy": best_energy,
            "time": elapsed_time,
            "energy_history": energy_history,
            "method": "Genetic search with quantum-inspired operators",
            "feasibility": self._check_assignment_feasibility(assignment),
        }
        print(f"Population search completed in {elapsed_time:.3f} seconds")
        print(f"Objective value: {results['objective']:.2f}")
        print(f"Best QUBO energy: {best_energy:.2f}")
        print(f"Assigned SKUs: {len(assignment)}")
        return results

    def _generate_random_solution(self) -> np.ndarray:
        """Generate a binary state with one location selected per SKU."""
        if self.qubo_matrix is None:
            raise RuntimeError("Formulate the QUBO before generating a state.")
        num_locations = len(self.locations)
        solution = np.zeros(self.qubo_matrix.shape[0], dtype=int)
        for i in range(len(self.skus)):
            location_index = np.random.randint(0, num_locations)
            solution[i * num_locations + location_index] = 1
        return solution

    def _tournament_selection(
        self, population: List[np.ndarray], energies: List[float]
    ) -> np.ndarray:
        """Select the lowest-energy member of a small random tournament."""
        tournament_size = min(3, len(population))
        candidates = np.random.choice(
            len(population), tournament_size, replace=False
        )
        best_index = min(candidates, key=lambda index: energies[index])
        return population[best_index].copy()

    def _quantum_crossover(
        self, parent1: np.ndarray, parent2: np.ndarray
    ) -> np.ndarray:
        """Combine parent bits and restore one selected location per SKU."""
        child = np.where(np.random.random(len(parent1)) < 0.5, parent1, parent2)
        num_locations = len(self.locations)

        for i in range(len(self.skus)):
            start = i * num_locations
            end = start + num_locations
            selected = np.flatnonzero(child[start:end])
            if len(selected) != 1:
                child[start:end] = 0
                location_index = int(np.random.randint(0, num_locations))
                child[start + location_index] = 1

        return child.astype(int)

    def _quantum_mutation(self, solution: np.ndarray) -> np.ndarray:
        """Randomly change selected SKU locations while preserving assignment."""
        mutated = solution.copy()
        mutation_rate = 0.1
        num_locations = len(self.locations)

        for i in range(len(self.skus)):
            if np.random.random() < mutation_rate:
                start = i * num_locations
                mutated[start : start + num_locations] = 0
                location_index = int(np.random.randint(0, num_locations))
                mutated[start + location_index] = 1

        return mutated

    def compare_methods(self) -> Dict[str, Any]:
        """Run the approaches and summarize their modeled outcomes."""
        print("\n=== METHOD COMPARISON ===")
        classical_results = self.classical_baseline()
        annealing_results = self.quantum_annealing_simulation()
        hybrid_results = self.hybrid_quantum_classical()
        inspired_results = self.quantum_inspired_algorithm()

        annealing_objective = self._calculate_objective(
            annealing_results["solution"]
        )
        baseline_objective = classical_results["objective"]
        denominator = abs(baseline_objective)

        def improvement(objective: float) -> float:
            if denominator == 0:
                return 0.0
            return (objective - baseline_objective) / denominator * 100.0

        improvements = {
            "quantum": improvement(annealing_objective),
            "hybrid": improvement(hybrid_results["objective"]),
            "quantum_inspired": improvement(inspired_results["objective"]),
        }

        def runtime_ratio(method_time: float) -> float:
            if method_time <= 0:
                return 0.0
            return classical_results["time"] / method_time

        comparison = {
            "classical": classical_results,
            "quantum": annealing_results,
            "hybrid": hybrid_results,
            "quantum_inspired": inspired_results,
            "improvements": improvements,
            "speedup": {
                "quantum_vs_classical": runtime_ratio(annealing_results["time"]),
                "hybrid_vs_classical": runtime_ratio(hybrid_results["time"]),
                "quantum_inspired_vs_classical": runtime_ratio(
                    inspired_results["time"]
                ),
            },
            "objectives": {
                "classical": baseline_objective,
                "quantum": annealing_objective,
                "hybrid": hybrid_results["objective"],
                "quantum_inspired": inspired_results["objective"],
            },
        }

        print(f"\nGreedy baseline objective: {baseline_objective:.2f}")
        print(f"Annealing-simulation improvement: {improvements['quantum']:.1f}%")
        print(f"Hybrid-procedure improvement: {improvements['hybrid']:.1f}%")
        print(
            "Population-search improvement: "
            f"{improvements['quantum_inspired']:.1f}%"
        )
        print("Runtime ratios are implementation-specific, not quantum speedups.")
        return comparison

    def visualize_quantum_results(self, comparison: Dict[str, Any]) -> None:
        """Plot objective values, runtimes, and recorded energy histories."""
        print("\n=== METHOD COMPARISON VISUALIZATION ===")
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))

        methods = ["Greedy", "Annealing simulation", "Hybrid", "Population search"]
        colors = ["lightblue", "lightgreen", "lightyellow", "lightcoral"]
        objectives = [
            comparison["objectives"]["classical"],
            comparison["objectives"]["quantum"],
            comparison["objectives"]["hybrid"],
            comparison["objectives"]["quantum_inspired"],
        ]
        bars = axes[0, 0].bar(methods, objectives, color=colors)
        axes[0, 0].set_title("Decoded Objective Value by Method")
        axes[0, 0].set_ylabel("Objective value (score units)")
        axes[0, 0].tick_params(axis="x", rotation=20)
        axes[0, 0].grid(True, alpha=0.3)
        for bar, objective in zip(bars, objectives):
            axes[0, 0].text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height(),
                f"{objective:.1f}",
                ha="center",
                va="bottom",
            )

        runtimes = [
            comparison["classical"]["time"],
            comparison["quantum"]["time"],
            comparison["hybrid"]["time"],
            comparison["quantum_inspired"]["time"],
        ]
        axes[0, 1].bar(methods, runtimes, color=colors)
        axes[0, 1].set_title("Measured Runtime by Method")
        axes[0, 1].set_ylabel("Elapsed time (seconds)")
        axes[0, 1].tick_params(axis="x", rotation=20)
        axes[0, 1].grid(True, alpha=0.3)

        improvements = [
            0.0,
            comparison["improvements"]["quantum"],
            comparison["improvements"]["hybrid"],
            comparison["improvements"]["quantum_inspired"],
        ]
        axes[0, 2].bar(methods, improvements, color=colors)
        axes[0, 2].set_title("Objective Change Relative to Greedy")
        axes[0, 2].set_ylabel("Objective change (percent)")
        axes[0, 2].tick_params(axis="x", rotation=20)
        axes[0, 2].grid(True, alpha=0.3)

        annealing_history = comparison["quantum"]["energy_history"]
        axes[1, 0].plot(
            range(1, len(annealing_history) + 1),
            annealing_history,
            color="green",
            linewidth=1.5,
        )
        axes[1, 0].set_title("Best-So-Far Annealing Energy")
        axes[1, 0].set_xlabel("Read")
        axes[1, 0].set_ylabel("QUBO energy")
        axes[1, 0].grid(True, alpha=0.3)

        population_history = comparison["quantum_inspired"]["energy_history"]
        axes[1, 1].plot(
            range(1, len(population_history) + 1),
            population_history,
            color="blue",
            linewidth=1.5,
        )
        axes[1, 1].set_title("Best-So-Far Population-Search Energy")
        axes[1, 1].set_xlabel("Generation")
        axes[1, 1].set_ylabel("QUBO energy")
        axes[1, 1].grid(True, alpha=0.3)

        ratio_labels = ["Annealing", "Hybrid", "Population"]
        ratios = [
            comparison["speedup"]["quantum_vs_classical"],
            comparison["speedup"]["hybrid_vs_classical"],
            comparison["speedup"]["quantum_inspired_vs_classical"],
        ]
        axes[1, 2].bar(ratio_labels, ratios, color=colors[1:])
        axes[1, 2].set_title("Greedy Runtime Divided by Method Runtime")
        axes[1, 2].set_ylabel("Runtime ratio (dimensionless)")
        axes[1, 2].grid(True, alpha=0.3)

        fig.suptitle(
            "Classical Implementations of QUBO-Based Slotting Searches",
            fontsize=14,
        )
        fig.tight_layout()
        plt.show()

    def analyze_quantum_advantage(
        self, comparison: Dict[str, Any]
    ) -> Dict[str, Any]:
        """Report heuristic thresholds without claiming quantum advantage."""
        print("\n=== INTERPRETING THE COMPARISON ===")
        improvement_threshold = 10.0
        runtime_ratio_threshold = 2.0
        annealing_improvement = comparison["improvements"]["quantum"]
        annealing_runtime_ratio = comparison["speedup"][
            "quantum_vs_classical"
        ]
        threshold_indicators = {
            "objective_threshold_met": (
                annealing_improvement > improvement_threshold
            ),
            "runtime_ratio_threshold_met": (
                annealing_runtime_ratio > runtime_ratio_threshold
            ),
        }
        assessment = {
            "quantum_advantage_established": False,
            "classical_simulation_threshold_indicators": threshold_indicators,
            "annealing_objective_change_percent": annealing_improvement,
            "annealing_runtime_ratio": annealing_runtime_ratio,
            "thresholds": {
                "objective_change_percent": improvement_threshold,
                "runtime_ratio": runtime_ratio_threshold,
            },
        }

        print(
            "Annealing-simulation objective change: "
            f"{annealing_improvement:.1f}% "
            f"(illustrative threshold: {improvement_threshold:.1f}%)"
        )
        print(
            "Greedy-to-simulation runtime ratio: "
            f"{annealing_runtime_ratio:.2f} "
            f"(illustrative threshold: {runtime_ratio_threshold:.1f})"
        )
        print(
            "These thresholds describe this comparison only; they do not "
            "establish quantum advantage. No quantum hardware is used."
        )
        return assessment

In [ ]:
# Initialize the main illustrative instance and run the comparison.
quantum_optimizer = QuantumSlottingOptimizer(num_skus=150, num_locations=30)
print("=== WAREHOUSE SLOTTING SEARCH COMPARISON ===")
comparison_results = quantum_optimizer.compare_methods()

# Visualize the recorded outcomes and inspect the comparison criteria.
quantum_optimizer.visualize_quantum_results(comparison_results)
quantum_advantage = quantum_optimizer.analyze_quantum_advantage(
    comparison_results
)

print("\n=== RUN SUMMARY ===")
print(
    f"Problem size: {len(quantum_optimizer.skus)} SKUs, "
    f"{len(quantum_optimizer.locations)} locations"
)
print(f"QUBO binary variables: {quantum_optimizer.qubo_matrix.shape[0]}")
for method_name in ["classical", "quantum", "hybrid", "quantum_inspired"]:
    method_results = comparison_results[method_name]
    if "feasibility" in method_results:
        feasibility = method_results["feasibility"]
    else:
        feasibility = quantum_optimizer._check_assignment_feasibility(
            method_results["solution"]
        )
    print(
        f"{method_name}: assigned {len(method_results['solution'])} SKUs; "
        f"aggregate-capacity feasible = {feasibility['feasible']}"
    )

## Results interpretation

The comparison cell computes results from the generated instance when the notebook is run. No numerical outcome is assumed here. Use the printed values and plots to answer the following questions:

1. **Objective value:** Which decoded assignment has the largest velocity-and-zone score? A larger score is better under this objective, but it does not by itself establish capacity feasibility.
2. **Feasibility:** Does each method assign every SKU, and does its assignment satisfy the aggregate space and weight limits? The QUBO's pairwise conflict penalties are only a surrogate for these aggregate limits.
3. **Energy versus objective:** QUBO energy includes both score terms and penalties. A low energy is not directly comparable to a high decoded objective; they are different quantities.
4. **Runtime:** The ratios compare measured runtimes of the notebook's classical implementations. They depend on the computer, software, instance, and implementation, and are not quantum speedup measurements.
5. **Search histories:** The annealing curve records the best energy observed up to each read. The population curve records the best energy observed up to each generation. A curve that levels off indicates that the recorded best value did not improve during that interval; it does not prove that a global optimum has been reached.

### Interpreting the method labels

The function names preserve the notebook's teaching sequence, but the computational procedures should be identified precisely. `quantum_annealing_simulation` performs a classical Metropolis-style simulated annealing search over QUBO bits. `quantum_inspired_algorithm` uses a classical population, tournament selection, crossover, and mutation; it is not QAOA or VQE. `hybrid_quantum_classical` combines the annealing-style simulation with classical local refinement. None of these procedures uses quantum hardware, and a favorable result on one generated instance does not demonstrate a quantum advantage.

### Strengths and limitations of the model

The QUBO makes the assignment score and selected penalties explicit in a single binary energy function, which supports study of annealing-style search. The greedy baseline gives a separate, interpretable construction that enforces aggregate location limits as it assigns SKUs. The local-refinement step illustrates how a search result can be improved through feasible single-SKU moves.

The model also has important limits. Pairwise conflict penalties do not guarantee aggregate capacity feasibility; the chosen penalty weights are modeling parameters, not universal guarantees; and decoding one location per SKU can conceal violations present in a raw binary state. The reported feasibility checks make aggregate violations visible after decoding, but they do not convert the QUBO into an exact aggregate-capacity formulation. The generated data and score are illustrative rather than operational measurements.

### Further study

A natural extension is to investigate how additional operational requirements—such as SKU-location eligibility, zone compatibility, or replenishment rules—would affect the model and its penalty terms. Larger instances also increase the dense matrix's memory requirements. Such extensions require a clearly specified model and should be evaluated with feasibility and objective measures appropriate to the added requirements.